# WC_BADGE_DETAILS_D ETL - ODI to Databricks Migration (PySpark)
### Badge Details Dimension - CUSTOM_TS_DW_BADGEDETAILS
**Source Table:** `workspace.PRXBI_TS.wc_mercury_badge_ts`
**Target Table:** `workspace.PRXBI_DW.wc_badge_details_d`
**Detection Strategy:** NOT_EXISTS (full column CDC with 45+ columns)
**DATASOURCE_NUM_ID:** 380

#### ODI Task Mapping (27 tasks)
| ODI Task | Description | PySpark Cell |
|----------|-------------|-------------|
| Tasks 1-6 | Variable Assignments (ETL Parameters) | Cells 2-4 |
| Task 10 | MAP_MAIN (serial container) | Grouping marker |
| Task 20 | EU (serial container) | Grouping marker |
| Task 30 | Drop C$ staging table | Cell 5 |
| Task 40 | Create C$ staging table (43 columns) | Cell 6 |
| Task 50 | Extract source data into C$ | Cell 7 |
| Task 60 | Analyze C$ table (DBMS_STATS) | Cell 8 |
| Task 70 | EU (serial container) | Grouping marker |
| Task 80 | Drop I$ flow table | Cell 9 |
| Task 90 | Create I$ flow table (52 columns) | Cell 10 |
| Task 100 | Insert into I$ (NOT_EXISTS detection) | Cell 11 |
| Task 110 | Create index on I$ | Skipped (Delta handles) |
| Task 120 | Analyze I$ (DBMS_STATS) | Cell 12 |
| Task 130 | Flag rows for update | Cell 13 |
| Task 140 | Flag useless rows | Skipped (NOT_EXISTS) |
| Tasks 150-160 | Update + Insert into target | Cell 14 (MERGE) |
| Task 170 | Commit | Implicit in Databricks |
| Task 180 | Drop I$ flow table | Cell 15 |
| Task 190 | MAP_CLEANUP (serial container) | Grouping marker |
| Task 200 | EU (serial container) | Grouping marker |
| Task 210 | Drop C$ staging table | Cell 16 |

#### Migration Notes
- Oracle `TS_STAGE` schema mapped to `workspace.PRXBI_TS`
- Oracle `DW_BIAPPS11G` schema mapped to `workspace.PRXBI_DW`
- `SYSTIMESTAMP` replaced with `current_timestamp()`
- `NVL2(x,'Y','N')` replaced with `CASE WHEN x IS NOT NULL THEN 'Y' ELSE 'N' END`
- Oracle `/*+ append */` hints and `NOLOGGING` removed
- Oracle indexes removed (Delta handles via Z-ORDER)
- `DBMS_STATS` replaced with `OPTIMIZE` + `ZORDER`
- Oracle sequences (`SEQ.NEXTVAL`) replaced with identity column on target table
- Separate UPDATE + INSERT replaced with MERGE INTO
- NULL-safe comparison uses Spark `<=>` operator
- `VARCHAR2` -> `STRING`, `NUMBER` -> `DECIMAL`/`INT`, `TIMESTAMP(n)` -> `TIMESTAMP`
- All tables use Delta format
- PySpark notebook: all SQL executed via `spark.sql()`; parameters via `dbutils.widgets`

## Tasks 1-6: Variable Assignments (ETL Parameters)
Define widgets for ETL_JOB_TYPE, DATASOURCE_NUM_ID, ETL_PROC_WID.
Query `wc_etl_parameters` for extract time windows and ROW_WID.

In [ ]:
# Task 1: Define widgets for ETL parameters
dbutils.widgets.text("ETL_JOB_TYPE", "EOD", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "380", "Datasource Num ID")
dbutils.widgets.text("ETL_PROC_WID", "1", "ETL Process WID")

In [ ]:
# Task 1: Get widget values into Python variables
v_etl_job_type = dbutils.widgets.get("ETL_JOB_TYPE")
v_datasource_num_id = dbutils.widgets.get("DATASOURCE_NUM_ID")
v_etl_proc_wid = dbutils.widgets.get("ETL_PROC_WID")

print(f"ETL_JOB_TYPE: {v_etl_job_type}")
print(f"DATASOURCE_NUM_ID: {v_datasource_num_id}")
print(f"ETL_PROC_WID: {v_etl_proc_wid}")

In [ ]:
# Task 2: V_ETL_LAST_EXTRACT_TIME
row = spark.sql(f"""
    SELECT etl_last_extract_time
    FROM workspace.PRXBI_DW.wc_etl_parameters
    WHERE ETL_JOB_TYPE = '{v_etl_job_type}'
""").collect()
V_ETL_LAST_EXTRACT_TIME = str(row[0][0]) if row else '1900-01-01 00:00:00.000000'
print(f"V_ETL_LAST_EXTRACT_TIME: {V_ETL_LAST_EXTRACT_TIME}")

# Task 3: V_ETL_CURRENT_EXTRACT_TIME
row = spark.sql(f"""
    SELECT etl_current_extract_time
    FROM workspace.PRXBI_DW.wc_etl_parameters
    WHERE ETL_JOB_TYPE = '{v_etl_job_type}'
""").collect()
V_ETL_CURRENT_EXTRACT_TIME = str(row[0][0]) if row else '9999-12-31 23:59:59.000000'
print(f"V_ETL_CURRENT_EXTRACT_TIME: {V_ETL_CURRENT_EXTRACT_TIME}")

# Task 4: v_ETL_LAST_EXTRACT_TIME (duplicate of Task 2, same query)
row = spark.sql(f"""
    SELECT etl_last_extract_time
    FROM workspace.PRXBI_DW.wc_etl_parameters
    WHERE ETL_JOB_TYPE = '{v_etl_job_type}'
""").collect()
v_ETL_LAST_EXTRACT_TIME = str(row[0][0]) if row else '1900-01-01 00:00:00.000000'
print(f"v_ETL_LAST_EXTRACT_TIME: {v_ETL_LAST_EXTRACT_TIME}")

# Task 5: v_ETL_CURRENT_EXTRACT_TIME (duplicate of Task 3, same query)
row = spark.sql(f"""
    SELECT etl_current_extract_time
    FROM workspace.PRXBI_DW.wc_etl_parameters
    WHERE ETL_JOB_TYPE = '{v_etl_job_type}'
""").collect()
v_ETL_CURRENT_EXTRACT_TIME = str(row[0][0]) if row else '9999-12-31 23:59:59.000000'
print(f"v_ETL_CURRENT_EXTRACT_TIME: {v_ETL_CURRENT_EXTRACT_TIME}")

# Task 6: ETLProcWID
row = spark.sql(f"""
    SELECT ROW_WID
    FROM workspace.PRXBI_DW.wc_etl_parameters
    WHERE ETL_JOB_TYPE = '{v_etl_job_type}'
""").collect()
ETLProcWID = str(row[0][0]) if row else '0'
print(f"ETLProcWID: {ETLProcWID}")

## Task 10: MAP_MAIN (Serial Container)
## Task 20: EU (Serial Container)
These are ODI grouping markers. Execution continues sequentially below.

## Task 30: Drop C$ Staging Table

In [ ]:
# Task 30: Drop C$ staging table
spark.sql("DROP TABLE IF EXISTS workspace.PRXBI_DW.c_badge_details_stg")
print("C$ staging table dropped.")

## Task 40: Create C$ Staging Table (43 columns)

In [ ]:
# Task 40: Create C$ staging table with all 43 source badge columns
spark.sql("""
    CREATE TABLE workspace.PRXBI_DW.c_badge_details_stg (
        ID                              STRING,
        BADGELOCATION                   STRING,
        BADGETOKEN                      STRING,
        BADGEVERSION                    DECIMAL(5,0),
        CONTACTEMAIL                    STRING,
        CONTACTFIRSTNAME                STRING,
        CONTACTJOBTITLE                 STRING,
        CONTACTLASTNAME                 STRING,
        CONTACTPERSONRXMASTERID         STRING,
        CREATEDBYREGISTRATIONTYPE       STRING,
        CREATEDBYTYPE                   STRING,
        CULTURE                         STRING,
        CUSTOMERTYPE                    STRING,
        EVENTEDITIONGBSCODE             STRING,
        ISBADGEUPDATE                   STRING,
        MARKETINGPREFERENCESPROMPTREQU   STRING,
        ORGANISATIONCITY                STRING,
        ORGANISATIONCOUNTRYCODE         STRING,
        ORGANISATIONDISPLAYNAME         STRING,
        ORGANISATIONRXMASTERID          STRING,
        ORGANISATIONSTATE               STRING,
        PARTICIPATINGORGANISATIONID     STRING,
        PRODUCTCODE                     STRING,
        QRCODECONTENT                   STRING,
        REGISTRATIONID                  STRING,
        STATUS                          DECIMAL(10,0),
        SUPPORTSTAFFCOMPANYADDRESS      STRING,
        SUPPORTSTAFFCOMPANYNAME         STRING,
        SUPPORTSTAFFMOBILEPHONE         STRING,
        SUPPORTSTAFFREPORTSTO           STRING,
        SUPPORTSTAFFSTANDS              STRING,
        SUPPORTSTAFFUSERACCESS          STRING,
        VERSIONNUMBER                   DECIMAL(10,0),
        MOBILEPHONE                     STRING,
        FIRSTSCANNEDDATE                TIMESTAMP,
        LASTPRINTEDDATE                 TIMESTAMP,
        ACCESSVALIDITYMODIFIEDDATE      TIMESTAMP,
        CREATEDDATE                     TIMESTAMP,
        COMPANYPRODUCTCODE              STRING,
        PAYMENTSTATUS                   STRING,
        PHOTOKEY                        STRING,
        PHOTOSOURCE                     STRING,
        PHOTOSOURCETYPE                 STRING
    ) USING DELTA
""")
print("C$ staging table created with 43 columns.")

## Task 50: Extract Source Data into C$ (INSERT/SELECT)
ColTxt = SELECT from source `WC_MERCURY_BADGE_TS`, DefTxt = INSERT into C$ staging.
INNER JOIN subquery deduplicates by ID using MAX(INT_INSERT_DATE) and MAX(VERSIONNUMBER)
within the ETL extract time window.

In [ ]:
# Task 50: Extract source data into C$ staging table
result = spark.sql(f"""
    INSERT INTO workspace.PRXBI_DW.c_badge_details_stg
    SELECT
        AMERCURY_BADGE_TS.ID AS ID,
        AMERCURY_BADGE_TS.BADGELOCATION AS BADGELOCATION,
        AMERCURY_BADGE_TS.BADGETOKEN AS BADGETOKEN,
        AMERCURY_BADGE_TS.BADGEVERSION AS BADGEVERSION,
        AMERCURY_BADGE_TS.CONTACTEMAIL AS CONTACTEMAIL,
        AMERCURY_BADGE_TS.CONTACTFIRSTNAME AS CONTACTFIRSTNAME,
        AMERCURY_BADGE_TS.CONTACTJOBTITLE AS CONTACTJOBTITLE,
        AMERCURY_BADGE_TS.CONTACTLASTNAME AS CONTACTLASTNAME,
        AMERCURY_BADGE_TS.CONTACTPERSONRXMASTERID AS CONTACTPERSONRXMASTERID,
        AMERCURY_BADGE_TS.CREATEDBYREGISTRATIONTYPE AS CREATEDBYREGISTRATIONTYPE,
        AMERCURY_BADGE_TS.CREATEDBYTYPE AS CREATEDBYTYPE,
        AMERCURY_BADGE_TS.CULTURE AS CULTURE,
        AMERCURY_BADGE_TS.CUSTOMERTYPE AS CUSTOMERTYPE,
        AMERCURY_BADGE_TS.EVENTEDITIONGBSCODE AS EVENTEDITIONGBSCODE,
        AMERCURY_BADGE_TS.ISBADGEUPDATE AS ISBADGEUPDATE,
        AMERCURY_BADGE_TS.MARKETINGPREFERENCESPROMPTREQUIRED AS MARKETINGPREFERENCESPROMPTREQU,
        AMERCURY_BADGE_TS.ORGANISATIONCITY AS ORGANISATIONCITY,
        AMERCURY_BADGE_TS.ORGANISATIONCOUNTRYCODE AS ORGANISATIONCOUNTRYCODE,
        AMERCURY_BADGE_TS.ORGANISATIONDISPLAYNAME AS ORGANISATIONDISPLAYNAME,
        AMERCURY_BADGE_TS.ORGANISATIONRXMASTERID AS ORGANISATIONRXMASTERID,
        AMERCURY_BADGE_TS.ORGANISATIONSTATE AS ORGANISATIONSTATE,
        AMERCURY_BADGE_TS.PARTICIPATINGORGANISATIONID AS PARTICIPATINGORGANISATIONID,
        AMERCURY_BADGE_TS.PRODUCTCODE AS PRODUCTCODE,
        AMERCURY_BADGE_TS.QRCODECONTENT AS QRCODECONTENT,
        AMERCURY_BADGE_TS.REGISTRATIONID AS REGISTRATIONID,
        AMERCURY_BADGE_TS.STATUS AS STATUS,
        AMERCURY_BADGE_TS.SUPPORTSTAFFCOMPANYADDRESS AS SUPPORTSTAFFCOMPANYADDRESS,
        AMERCURY_BADGE_TS.SUPPORTSTAFFCOMPANYNAME AS SUPPORTSTAFFCOMPANYNAME,
        AMERCURY_BADGE_TS.SUPPORTSTAFFMOBILEPHONE AS SUPPORTSTAFFMOBILEPHONE,
        AMERCURY_BADGE_TS.SUPPORTSTAFFREPORTSTO AS SUPPORTSTAFFREPORTSTO,
        AMERCURY_BADGE_TS.SUPPORTSTAFFSTANDS AS SUPPORTSTAFFSTANDS,
        AMERCURY_BADGE_TS.SUPPORTSTAFFUSERACCESS AS SUPPORTSTAFFUSERACCESS,
        AMERCURY_BADGE_TS.VERSIONNUMBER AS VERSIONNUMBER,
        AMERCURY_BADGE_TS.MOBILEPHONE AS MOBILEPHONE,
        AMERCURY_BADGE_TS.FIRSTSCANNEDDATE AS FIRSTSCANNEDDATE,
        AMERCURY_BADGE_TS.LASTPRINTEDDATE AS LASTPRINTEDDATE,
        AMERCURY_BADGE_TS.ACCESSVALIDITYMODIFIEDDATE AS ACCESSVALIDITYMODIFIEDDATE,
        AMERCURY_BADGE_TS.CREATEDDATE AS CREATEDDATE,
        AMERCURY_BADGE_TS.COMPANYPRODUCTCODE AS COMPANYPRODUCTCODE,
        AMERCURY_BADGE_TS.PAYMENTSTATUS AS PAYMENTSTATUS,
        AMERCURY_BADGE_TS.PHOTOKEY AS PHOTOKEY,
        AMERCURY_BADGE_TS.PHOTOSOURCE AS PHOTOSOURCE,
        AMERCURY_BADGE_TS.PHOTOSOURCETYPE AS PHOTOSOURCETYPE
    FROM workspace.PRXBI_TS.wc_mercury_badge_ts AMERCURY_BADGE_TS
    INNER JOIN (
        SELECT
            AMERCURY_BADGE_TS_1.ID AS ID,
            MAX(AMERCURY_BADGE_TS_1.INT_INSERT_DATE) AS INT_INSERT_DATE,
            MAX(AMERCURY_BADGE_TS_1.VERSIONNUMBER) AS VERSIONNUMBER
        FROM workspace.PRXBI_TS.wc_mercury_badge_ts AMERCURY_BADGE_TS_1
        WHERE (AMERCURY_BADGE_TS_1.INT_INSERT_DATE > CAST('{V_ETL_LAST_EXTRACT_TIME}' AS TIMESTAMP)
            AND AMERCURY_BADGE_TS_1.INT_INSERT_DATE <= CAST('{V_ETL_CURRENT_EXTRACT_TIME}' AS TIMESTAMP))
        GROUP BY AMERCURY_BADGE_TS_1.ID
    ) AMERCURY_BADGE_TS_2
    ON AMERCURY_BADGE_TS.INT_INSERT_DATE = AMERCURY_BADGE_TS_2.INT_INSERT_DATE
        AND AMERCURY_BADGE_TS.VERSIONNUMBER = AMERCURY_BADGE_TS_2.VERSIONNUMBER
        AND AMERCURY_BADGE_TS.ID = AMERCURY_BADGE_TS_2.ID
    WHERE (1=1)
""")

stg_count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.PRXBI_DW.c_badge_details_stg").collect()[0][0]
print(f"C$ staging table loaded with {stg_count} records.")

## Task 60: Analyze C$ Table (DBMS_STATS -> OPTIMIZE)

In [ ]:
# Task 60: Analyze C$ staging table (replaces Oracle DBMS_STATS.GATHER_TABLE_STATS)
spark.sql("OPTIMIZE workspace.PRXBI_DW.c_badge_details_stg")
print("C$ staging table optimized.")

## Task 70: EU (Serial Container)
ODI grouping marker. Execution continues sequentially.

## Task 80: Drop I$ Flow Table

In [ ]:
# Task 80: Drop I$ flow table
spark.sql("DROP TABLE IF EXISTS workspace.PRXBI_DW.i_badge_details_flow")
print("I$ flow table dropped.")

## Task 90: Create I$ Flow Table (52 columns)

In [ ]:
# Task 90: Create I$ flow table with all 52 columns
spark.sql("""
    CREATE TABLE workspace.PRXBI_DW.i_badge_details_flow (
        ROW_WID                         DECIMAL(10,0),
        BADGE_ID                        STRING,
        BADGE_LOCATION                  STRING,
        BADGE_TOKEN                     STRING,
        BADGE_VERSION                   DECIMAL(5,0),
        CONTACT_EMAIL                   STRING,
        CONTACT_FIRST_NAME              STRING,
        CONTACT_LAST_NAME               STRING,
        CONTACT_JOB_TITLE               STRING,
        CONTACT_PERSON_ID               STRING,
        CREATION_REG_TYPE               STRING,
        CREATION_TYPE                   STRING,
        CULTURE                         STRING,
        CUSTOMER_TYPE                   STRING,
        EVENT_EDITION_CODE              STRING,
        BADGE_UPDATE_FLG                STRING,
        MARKETING_PREF_PROMPT           STRING,
        ORG_NAME                        STRING,
        ORG_CITY                        STRING,
        ORG_COUNTRY                     STRING,
        ORG_ID                          STRING,
        ORG_STATE                       STRING,
        PARTICIPATING_ORG_ID            STRING,
        PRODUCT_CODE                    STRING,
        QR_CODE                         STRING,
        REGISTRATION_ID                 STRING,
        STATUS                          DECIMAL(10,0),
        STAFF_COMPANY_NAME              STRING,
        STAFF_COMPANY_ADDR              STRING,
        STAFF_PHONE_NUM                 STRING,
        STAFF_REPORTING                 STRING,
        STAFF_STANDS                    STRING,
        STAFF_USER_ACCESS               STRING,
        VERSION_NUM                     DECIMAL(10,0),
        INTEGRATION_ID                  STRING,
        DATASOURCE_NUM_ID               STRING,
        W_INSERT_DT                     DATE,
        W_UPDATE_DT                     DATE,
        MOBILEPHONE                     STRING,
        FIRSTSCANNEDDATE                TIMESTAMP,
        LASTPRINTEDDATE                 TIMESTAMP,
        FIRSTSCANNEDDATE_FLG            STRING,
        LASTPRINTEDDATE_FLG             STRING,
        ACCESSVALIDITYMODIFIEDDATE      TIMESTAMP,
        CREATEDDATE                     TIMESTAMP,
        COMPANYPRODUCTCODE              STRING,
        PAYMENTSTATUS                   STRING,
        PHOTOKEY                        STRING,
        PHOTOSOURCE                     STRING,
        PHOTOSOURCETYPE                 STRING,
        PACKAGE_NAME                    STRING,
        IND_UPDATE                      STRING
    ) USING DELTA
""")
print("I$ flow table created with 52 columns.")

## Task 100: Insert into I$ Flow Table (NOT_EXISTS Detection)
Core transformation: column mappings from C$ to I$, LEFT OUTER JOIN to WC_BADGE_PRODUCT_D
for PACKAGE_NAME, and NOT EXISTS detection comparing all 45 columns with NULL-safe `<=>` checks.

In [ ]:
# Task 100: Insert into I$ flow table with column mappings, enrichment, and NOT EXISTS detection
result = spark.sql(f"""
    INSERT INTO workspace.PRXBI_DW.i_badge_details_flow
    (
        BADGE_ID, BADGE_LOCATION, BADGE_TOKEN, BADGE_VERSION,
        CONTACT_EMAIL, CONTACT_FIRST_NAME, CONTACT_LAST_NAME, CONTACT_JOB_TITLE,
        CONTACT_PERSON_ID, CREATION_REG_TYPE, CREATION_TYPE, CULTURE,
        CUSTOMER_TYPE, EVENT_EDITION_CODE, BADGE_UPDATE_FLG, MARKETING_PREF_PROMPT,
        ORG_NAME, ORG_CITY, ORG_COUNTRY, ORG_ID, ORG_STATE,
        PARTICIPATING_ORG_ID, PRODUCT_CODE, QR_CODE, REGISTRATION_ID, STATUS,
        STAFF_COMPANY_NAME, STAFF_COMPANY_ADDR, STAFF_PHONE_NUM,
        STAFF_REPORTING, STAFF_STANDS, STAFF_USER_ACCESS, VERSION_NUM,
        INTEGRATION_ID, DATASOURCE_NUM_ID,
        MOBILEPHONE, FIRSTSCANNEDDATE, LASTPRINTEDDATE,
        FIRSTSCANNEDDATE_FLG, LASTPRINTEDDATE_FLG,
        ACCESSVALIDITYMODIFIEDDATE, CREATEDDATE,
        COMPANYPRODUCTCODE, PAYMENTSTATUS, PHOTOKEY, PHOTOSOURCE, PHOTOSOURCETYPE,
        PACKAGE_NAME, IND_UPDATE
    )
    SELECT
        JOIN1_A.ID                                  AS BADGE_ID,
        JOIN1_A.BADGELOCATION                       AS BADGE_LOCATION,
        JOIN1_A.BADGETOKEN                          AS BADGE_TOKEN,
        JOIN1_A.BADGEVERSION                        AS BADGE_VERSION,
        JOIN1_A.CONTACTEMAIL                        AS CONTACT_EMAIL,
        JOIN1_A.CONTACTFIRSTNAME                    AS CONTACT_FIRST_NAME,
        JOIN1_A.CONTACTLASTNAME                     AS CONTACT_LAST_NAME,
        JOIN1_A.CONTACTJOBTITLE                     AS CONTACT_JOB_TITLE,
        JOIN1_A.CONTACTPERSONRXMASTERID             AS CONTACT_PERSON_ID,
        JOIN1_A.CREATEDBYREGISTRATIONTYPE            AS CREATION_REG_TYPE,
        JOIN1_A.CREATEDBYTYPE                       AS CREATION_TYPE,
        JOIN1_A.CULTURE                             AS CULTURE,
        JOIN1_A.CUSTOMERTYPE                        AS CUSTOMER_TYPE,
        JOIN1_A.EVENTEDITIONGBSCODE                 AS EVENT_EDITION_CODE,
        JOIN1_A.ISBADGEUPDATE                       AS BADGE_UPDATE_FLG,
        JOIN1_A.MARKETINGPREFERENCESPROMPTREQU       AS MARKETING_PREF_PROMPT,
        JOIN1_A.ORGANISATIONDISPLAYNAME              AS ORG_NAME,
        JOIN1_A.ORGANISATIONCITY                    AS ORG_CITY,
        JOIN1_A.ORGANISATIONCOUNTRYCODE              AS ORG_COUNTRY,
        JOIN1_A.ORGANISATIONRXMASTERID               AS ORG_ID,
        JOIN1_A.ORGANISATIONSTATE                   AS ORG_STATE,
        JOIN1_A.PARTICIPATINGORGANISATIONID          AS PARTICIPATING_ORG_ID,
        JOIN1_A.PRODUCTCODE                         AS PRODUCT_CODE,
        JOIN1_A.QRCODECONTENT                       AS QR_CODE,
        JOIN1_A.REGISTRATIONID                      AS REGISTRATION_ID,
        JOIN1_A.STATUS                              AS STATUS,
        JOIN1_A.SUPPORTSTAFFCOMPANYNAME              AS STAFF_COMPANY_NAME,
        JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS           AS STAFF_COMPANY_ADDR,
        JOIN1_A.SUPPORTSTAFFMOBILEPHONE              AS STAFF_PHONE_NUM,
        JOIN1_A.SUPPORTSTAFFREPORTSTO                AS STAFF_REPORTING,
        JOIN1_A.SUPPORTSTAFFSTANDS                  AS STAFF_STANDS,
        JOIN1_A.SUPPORTSTAFFUSERACCESS               AS STAFF_USER_ACCESS,
        JOIN1_A.VERSIONNUMBER                       AS VERSION_NUM,
        JOIN1_A.ID                                  AS INTEGRATION_ID,
        '380'                                       AS DATASOURCE_NUM_ID,
        JOIN1_A.MOBILEPHONE                         AS MOBILEPHONE,
        JOIN1_A.FIRSTSCANNEDDATE                    AS FIRSTSCANNEDDATE,
        JOIN1_A.LASTPRINTEDDATE                     AS LASTPRINTEDDATE,
        CASE WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
        CASE WHEN JOIN1_A.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END  AS LASTPRINTEDDATE_FLG,
        JOIN1_A.ACCESSVALIDITYMODIFIEDDATE           AS ACCESSVALIDITYMODIFIEDDATE,
        JOIN1_A.CREATEDDATE                         AS CREATEDDATE,
        JOIN1_A.COMPANYPRODUCTCODE                  AS COMPANYPRODUCTCODE,
        JOIN1_A.PAYMENTSTATUS                       AS PAYMENTSTATUS,
        JOIN1_A.PHOTOKEY                            AS PHOTOKEY,
        JOIN1_A.PHOTOSOURCE                         AS PHOTOSOURCE,
        JOIN1_A.PHOTOSOURCETYPE                     AS PHOTOSOURCETYPE,
        WC_BADGE_PRODUCT_D_2.NAME_1                 AS PACKAGE_NAME,
        'I'                                         AS IND_UPDATE
    FROM workspace.PRXBI_DW.c_badge_details_stg JOIN1_A
    LEFT OUTER JOIN (
        SELECT
            ID,
            SKU,
            NAME,
            RANK() OVER (PARTITION BY SKU ORDER BY ID DESC) AS COL,
            SKU AS SKU_1,
            NAME AS NAME_1
        FROM workspace.PRXBI_DW.wc_badge_product_d
    ) WC_BADGE_PRODUCT_D_2
    ON JOIN1_A.PRODUCTCODE = WC_BADGE_PRODUCT_D_2.SKU_1
        AND WC_BADGE_PRODUCT_D_2.COL = 1
    WHERE NOT EXISTS (
        SELECT 1
        FROM workspace.PRXBI_DW.wc_badge_details_d T
        WHERE T.INTEGRATION_ID <=> JOIN1_A.ID
            AND T.DATASOURCE_NUM_ID <=> '380'
            AND T.BADGE_ID <=> JOIN1_A.ID
            AND T.BADGE_LOCATION <=> JOIN1_A.BADGELOCATION
            AND T.BADGE_TOKEN <=> JOIN1_A.BADGETOKEN
            AND T.BADGE_VERSION <=> JOIN1_A.BADGEVERSION
            AND T.CONTACT_EMAIL <=> JOIN1_A.CONTACTEMAIL
            AND T.CONTACT_FIRST_NAME <=> JOIN1_A.CONTACTFIRSTNAME
            AND T.CONTACT_LAST_NAME <=> JOIN1_A.CONTACTLASTNAME
            AND T.CONTACT_JOB_TITLE <=> JOIN1_A.CONTACTJOBTITLE
            AND T.CONTACT_PERSON_ID <=> JOIN1_A.CONTACTPERSONRXMASTERID
            AND T.CREATION_REG_TYPE <=> JOIN1_A.CREATEDBYREGISTRATIONTYPE
            AND T.CREATION_TYPE <=> JOIN1_A.CREATEDBYTYPE
            AND T.CULTURE <=> JOIN1_A.CULTURE
            AND T.CUSTOMER_TYPE <=> JOIN1_A.CUSTOMERTYPE
            AND T.EVENT_EDITION_CODE <=> JOIN1_A.EVENTEDITIONGBSCODE
            AND T.BADGE_UPDATE_FLG <=> JOIN1_A.ISBADGEUPDATE
            AND T.MARKETING_PREF_PROMPT <=> JOIN1_A.MARKETINGPREFERENCESPROMPTREQU
            AND T.ORG_NAME <=> JOIN1_A.ORGANISATIONDISPLAYNAME
            AND T.ORG_CITY <=> JOIN1_A.ORGANISATIONCITY
            AND T.ORG_COUNTRY <=> JOIN1_A.ORGANISATIONCOUNTRYCODE
            AND T.ORG_ID <=> JOIN1_A.ORGANISATIONRXMASTERID
            AND T.ORG_STATE <=> JOIN1_A.ORGANISATIONSTATE
            AND T.PARTICIPATING_ORG_ID <=> JOIN1_A.PARTICIPATINGORGANISATIONID
            AND T.PRODUCT_CODE <=> JOIN1_A.PRODUCTCODE
            AND T.QR_CODE <=> JOIN1_A.QRCODECONTENT
            AND T.REGISTRATION_ID <=> JOIN1_A.REGISTRATIONID
            AND T.STATUS <=> JOIN1_A.STATUS
            AND T.STAFF_COMPANY_NAME <=> JOIN1_A.SUPPORTSTAFFCOMPANYNAME
            AND T.STAFF_COMPANY_ADDR <=> JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS
            AND T.STAFF_PHONE_NUM <=> JOIN1_A.SUPPORTSTAFFMOBILEPHONE
            AND T.STAFF_REPORTING <=> JOIN1_A.SUPPORTSTAFFREPORTSTO
            AND T.STAFF_STANDS <=> JOIN1_A.SUPPORTSTAFFSTANDS
            AND T.STAFF_USER_ACCESS <=> JOIN1_A.SUPPORTSTAFFUSERACCESS
            AND T.VERSION_NUM <=> JOIN1_A.VERSIONNUMBER
            AND T.MOBILEPHONE <=> JOIN1_A.MOBILEPHONE
            AND T.FIRSTSCANNEDDATE <=> JOIN1_A.FIRSTSCANNEDDATE
            AND T.LASTPRINTEDDATE <=> JOIN1_A.LASTPRINTEDDATE
            AND T.FIRSTSCANNEDDATE_FLG <=> CASE WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END
            AND T.LASTPRINTEDDATE_FLG <=> CASE WHEN JOIN1_A.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END
            AND T.ACCESSVALIDITYMODIFIEDDATE <=> JOIN1_A.ACCESSVALIDITYMODIFIEDDATE
            AND T.CREATEDDATE <=> JOIN1_A.CREATEDDATE
            AND T.COMPANYPRODUCTCODE <=> JOIN1_A.COMPANYPRODUCTCODE
            AND T.PAYMENTSTATUS <=> JOIN1_A.PAYMENTSTATUS
            AND T.PHOTOKEY <=> JOIN1_A.PHOTOKEY
            AND T.PHOTOSOURCE <=> JOIN1_A.PHOTOSOURCE
            AND T.PHOTOSOURCETYPE <=> JOIN1_A.PHOTOSOURCETYPE
            AND T.PACKAGE_NAME <=> WC_BADGE_PRODUCT_D_2.NAME_1
    )
""")

flow_count = spark.sql("SELECT COUNT(*) AS cnt FROM workspace.PRXBI_DW.i_badge_details_flow").collect()[0][0]
print(f"I$ flow table loaded with {flow_count} records.")

## Task 110: Create Index on I$ (INTEGRATION_ID, DATASOURCE_NUM_ID)
Skipped in Databricks - Delta Lake handles data skipping natively.

## Task 120: Analyze I$ (DBMS_STATS -> OPTIMIZE + ZORDER)

In [ ]:
# Task 120: Analyze I$ flow table (replaces Oracle DBMS_STATS.GATHER_TABLE_STATS)
spark.sql("""
    OPTIMIZE workspace.PRXBI_DW.i_badge_details_flow
    ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID)
""")
print("I$ flow table optimized with ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID).")

## Task 130: Flag Rows for Update
Set IND_UPDATE = 'U' where (INTEGRATION_ID, DATASOURCE_NUM_ID) already exists in target.

In [ ]:
# Task 130: Flag rows for update
spark.sql("""
    UPDATE workspace.PRXBI_DW.i_badge_details_flow I
    SET IND_UPDATE = 'U'
    WHERE (I.INTEGRATION_ID, I.DATASOURCE_NUM_ID) IN (
        SELECT INTEGRATION_ID, DATASOURCE_NUM_ID
        FROM workspace.PRXBI_DW.wc_badge_details_d
    )
""")

# Display IND_UPDATE breakdown
breakdown = spark.sql("""
    SELECT IND_UPDATE, COUNT(*) AS record_count
    FROM workspace.PRXBI_DW.i_badge_details_flow
    GROUP BY IND_UPDATE
    ORDER BY IND_UPDATE
""").collect()
for row in breakdown:
    print(f"IND_UPDATE='{row['IND_UPDATE']}': {row['record_count']} records")

## Task 140: Flag Useless Rows
Skipped - NOT_EXISTS detection strategy already filters unchanged rows.

## Tasks 150-160: Update Existing Rows + Insert New Rows (MERGE INTO)
Replaces separate Oracle UPDATE (Task 150) and INSERT (Task 160) with a single MERGE.
- WHEN MATCHED (IND_UPDATE='U'): Update all 46 columns + W_UPDATE_DT = current_timestamp()
- WHEN NOT MATCHED (IND_UPDATE='I'): Insert all 48+ columns with W_INSERT_DT and W_UPDATE_DT
- ROW_WID is handled by identity column on target table (replaces WC_BADGE_DETAILS_D_SEQ.NEXTVAL)

In [ ]:
# Tasks 150-160: MERGE INTO target from I$ flow table
spark.sql("""
    MERGE INTO workspace.PRXBI_DW.wc_badge_details_d T
    USING workspace.PRXBI_DW.i_badge_details_flow S
    ON T.INTEGRATION_ID = S.INTEGRATION_ID
        AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID

    WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
        T.BADGE_ID                      = S.BADGE_ID,
        T.BADGE_LOCATION                = S.BADGE_LOCATION,
        T.BADGE_TOKEN                   = S.BADGE_TOKEN,
        T.BADGE_VERSION                 = S.BADGE_VERSION,
        T.CONTACT_EMAIL                 = S.CONTACT_EMAIL,
        T.CONTACT_FIRST_NAME            = S.CONTACT_FIRST_NAME,
        T.CONTACT_LAST_NAME             = S.CONTACT_LAST_NAME,
        T.CONTACT_JOB_TITLE             = S.CONTACT_JOB_TITLE,
        T.CONTACT_PERSON_ID             = S.CONTACT_PERSON_ID,
        T.CREATION_REG_TYPE             = S.CREATION_REG_TYPE,
        T.CREATION_TYPE                 = S.CREATION_TYPE,
        T.CULTURE                       = S.CULTURE,
        T.CUSTOMER_TYPE                 = S.CUSTOMER_TYPE,
        T.EVENT_EDITION_CODE            = S.EVENT_EDITION_CODE,
        T.BADGE_UPDATE_FLG              = S.BADGE_UPDATE_FLG,
        T.MARKETING_PREF_PROMPT         = S.MARKETING_PREF_PROMPT,
        T.ORG_NAME                      = S.ORG_NAME,
        T.ORG_CITY                      = S.ORG_CITY,
        T.ORG_COUNTRY                   = S.ORG_COUNTRY,
        T.ORG_ID                        = S.ORG_ID,
        T.ORG_STATE                     = S.ORG_STATE,
        T.PARTICIPATING_ORG_ID          = S.PARTICIPATING_ORG_ID,
        T.PRODUCT_CODE                  = S.PRODUCT_CODE,
        T.QR_CODE                       = S.QR_CODE,
        T.REGISTRATION_ID               = S.REGISTRATION_ID,
        T.STATUS                        = S.STATUS,
        T.STAFF_COMPANY_NAME            = S.STAFF_COMPANY_NAME,
        T.STAFF_COMPANY_ADDR            = S.STAFF_COMPANY_ADDR,
        T.STAFF_PHONE_NUM               = S.STAFF_PHONE_NUM,
        T.STAFF_REPORTING               = S.STAFF_REPORTING,
        T.STAFF_STANDS                  = S.STAFF_STANDS,
        T.STAFF_USER_ACCESS             = S.STAFF_USER_ACCESS,
        T.VERSION_NUM                   = S.VERSION_NUM,
        T.MOBILEPHONE                   = S.MOBILEPHONE,
        T.FIRSTSCANNEDDATE              = S.FIRSTSCANNEDDATE,
        T.LASTPRINTEDDATE               = S.LASTPRINTEDDATE,
        T.FIRSTSCANNEDDATE_FLG          = S.FIRSTSCANNEDDATE_FLG,
        T.LASTPRINTEDDATE_FLG           = S.LASTPRINTEDDATE_FLG,
        T.ACCESSVALIDITYMODIFIEDDATE    = S.ACCESSVALIDITYMODIFIEDDATE,
        T.CREATEDDATE                   = S.CREATEDDATE,
        T.COMPANYPRODUCTCODE            = S.COMPANYPRODUCTCODE,
        T.PAYMENTSTATUS                 = S.PAYMENTSTATUS,
        T.PHOTOKEY                      = S.PHOTOKEY,
        T.PHOTOSOURCE                   = S.PHOTOSOURCE,
        T.PHOTOSOURCETYPE               = S.PHOTOSOURCETYPE,
        T.PACKAGE_NAME                  = S.PACKAGE_NAME,
        T.W_UPDATE_DT                   = current_timestamp()

    WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
        BADGE_ID,
        BADGE_LOCATION,
        BADGE_TOKEN,
        BADGE_VERSION,
        CONTACT_EMAIL,
        CONTACT_FIRST_NAME,
        CONTACT_LAST_NAME,
        CONTACT_JOB_TITLE,
        CONTACT_PERSON_ID,
        CREATION_REG_TYPE,
        CREATION_TYPE,
        CULTURE,
        CUSTOMER_TYPE,
        EVENT_EDITION_CODE,
        BADGE_UPDATE_FLG,
        MARKETING_PREF_PROMPT,
        ORG_NAME,
        ORG_CITY,
        ORG_COUNTRY,
        ORG_ID,
        ORG_STATE,
        PARTICIPATING_ORG_ID,
        PRODUCT_CODE,
        QR_CODE,
        REGISTRATION_ID,
        STATUS,
        STAFF_COMPANY_NAME,
        STAFF_COMPANY_ADDR,
        STAFF_PHONE_NUM,
        STAFF_REPORTING,
        STAFF_STANDS,
        STAFF_USER_ACCESS,
        VERSION_NUM,
        INTEGRATION_ID,
        DATASOURCE_NUM_ID,
        W_INSERT_DT,
        W_UPDATE_DT,
        MOBILEPHONE,
        FIRSTSCANNEDDATE,
        LASTPRINTEDDATE,
        FIRSTSCANNEDDATE_FLG,
        LASTPRINTEDDATE_FLG,
        ACCESSVALIDITYMODIFIEDDATE,
        CREATEDDATE,
        COMPANYPRODUCTCODE,
        PAYMENTSTATUS,
        PHOTOKEY,
        PHOTOSOURCE,
        PHOTOSOURCETYPE,
        PACKAGE_NAME
    ) VALUES (
        S.BADGE_ID,
        S.BADGE_LOCATION,
        S.BADGE_TOKEN,
        S.BADGE_VERSION,
        S.CONTACT_EMAIL,
        S.CONTACT_FIRST_NAME,
        S.CONTACT_LAST_NAME,
        S.CONTACT_JOB_TITLE,
        S.CONTACT_PERSON_ID,
        S.CREATION_REG_TYPE,
        S.CREATION_TYPE,
        S.CULTURE,
        S.CUSTOMER_TYPE,
        S.EVENT_EDITION_CODE,
        S.BADGE_UPDATE_FLG,
        S.MARKETING_PREF_PROMPT,
        S.ORG_NAME,
        S.ORG_CITY,
        S.ORG_COUNTRY,
        S.ORG_ID,
        S.ORG_STATE,
        S.PARTICIPATING_ORG_ID,
        S.PRODUCT_CODE,
        S.QR_CODE,
        S.REGISTRATION_ID,
        S.STATUS,
        S.STAFF_COMPANY_NAME,
        S.STAFF_COMPANY_ADDR,
        S.STAFF_PHONE_NUM,
        S.STAFF_REPORTING,
        S.STAFF_STANDS,
        S.STAFF_USER_ACCESS,
        S.VERSION_NUM,
        S.INTEGRATION_ID,
        S.DATASOURCE_NUM_ID,
        current_timestamp(),
        current_timestamp(),
        S.MOBILEPHONE,
        S.FIRSTSCANNEDDATE,
        S.LASTPRINTEDDATE,
        S.FIRSTSCANNEDDATE_FLG,
        S.LASTPRINTEDDATE_FLG,
        S.ACCESSVALIDITYMODIFIEDDATE,
        S.CREATEDDATE,
        S.COMPANYPRODUCTCODE,
        S.PAYMENTSTATUS,
        S.PHOTOKEY,
        S.PHOTOSOURCE,
        S.PHOTOSOURCETYPE,
        S.PACKAGE_NAME
    )
""")
print("MERGE INTO target table completed (Tasks 150-160).")

## Task 170: Commit
Implicit in Databricks Delta Lake - each SQL statement is auto-committed.

## Task 180: Drop I$ Flow Table

In [ ]:
# Task 180: Drop I$ flow table
spark.sql("DROP TABLE IF EXISTS workspace.PRXBI_DW.i_badge_details_flow")
print("I$ flow table dropped (Task 180).")

## Task 190: MAP_CLEANUP (Serial Container)
## Task 200: EU (Serial Container)
ODI grouping markers. Execution continues sequentially.

## Task 210: Drop C$ Staging Table (Final Cleanup)

In [ ]:
# Task 210: Drop C$ staging table (final cleanup)
spark.sql("DROP TABLE IF EXISTS workspace.PRXBI_DW.c_badge_details_stg")
print("C$ staging table dropped (Task 210).")

## Post-ETL: Optimize Target Table

In [ ]:
# Post-ETL: Optimize target table with ZORDER for query performance
spark.sql("""
    OPTIMIZE workspace.PRXBI_DW.wc_badge_details_d
    ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID)
""")
print("Target table optimized with ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID).")

## Validation

In [ ]:
# Final validation: display target table statistics
validation_df = spark.sql(f"""
    SELECT
        COUNT(*)                        AS total_records,
        COUNT(DISTINCT INTEGRATION_ID)  AS distinct_badges,
        MIN(W_INSERT_DT)                AS earliest_insert,
        MAX(W_UPDATE_DT)                AS latest_update,
        SUM(CASE WHEN W_UPDATE_DT > W_INSERT_DT THEN 1 ELSE 0 END) AS updated_records
    FROM workspace.PRXBI_DW.wc_badge_details_d
    WHERE DATASOURCE_NUM_ID = '{v_datasource_num_id}'
""")
validation_df.show(truncate=False)
print("ETL job completed successfully.")

## Conversion Notes (ODI to Databricks PySpark)

| ODI / Oracle Construct | Databricks / PySpark Equivalent |
|---|---|
| `TS_STAGE` schema | `workspace.PRXBI_TS` |
| `DW_BIAPPS11G` schema | `workspace.PRXBI_DW` |
| `SYSTIMESTAMP` | `current_timestamp()` |
| `NVL2(x,'Y','N')` | `CASE WHEN x IS NOT NULL THEN 'Y' ELSE 'N' END` |
| `TO_TIMESTAMP('#GLOBAL.V','YYYY-MM-DD HH24:MI:SS.FF')` | Python f-string with `CAST('{var}' AS TIMESTAMP)` |
| `/*+ append */` hint | Removed (Delta handles append natively) |
| `NOLOGGING` | Removed (Delta manages transaction logs) |
| Oracle indexes | Removed (Delta uses Z-ORDER for data skipping) |
| `DBMS_STATS.GATHER_TABLE_STATS` | `OPTIMIZE` + `ZORDER BY` |
| `WC_BADGE_DETAILS_D_SEQ.NEXTVAL` | Identity column on target table |
| `#GLOBAL.v_ETL_JOB_TYPE` | `dbutils.widgets.get('ETL_JOB_TYPE')` |
| Separate UPDATE + INSERT | `MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED` |
| `VARCHAR2` | `STRING` |
| `NUMBER(x,y)` | `DECIMAL(x,y)` or `INT` |
| `TIMESTAMP(6)` / `TIMESTAMP(7)` | `TIMESTAMP` |
| `(T.COL = S.COL) OR (T.COL IS NULL AND S.COL IS NULL)` | `T.COL <=> S.COL` (NULL-safe equality) |
| C$ table | `workspace.PRXBI_DW.c_badge_details_stg` (Delta) |
| I$ table | `workspace.PRXBI_DW.i_badge_details_flow` (Delta) |
| SQL `%sql` magic | `spark.sql()` in PySpark |
| `CREATE WIDGET TEXT` | `dbutils.widgets.text()` |
| `${widget}` reference | `dbutils.widgets.get()` + f-string |